# LaLiga football match forecasting dataset

This notebook downloads the reproducible v0.2.0 match-level foundation, builds leakage-safe pre-match features, adds the available StatsBomb historical LaLiga lineups, events, players, and managers, and evaluates a small baseline. Publishing is optional: by default, the generated artifacts stay in the Colab runtime. To publish your own copy, set `PUBLISH_TO_HUB = True` in the optional publishing cell and configure a write-capable `HF_TOKEN` Colab Secret. Keep execution outputs cleared before committing or sharing the notebook.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir('/content')
repo_url = 'https://github.com/EF-Code/laliga-match-forecasting.git'
repo_revision = '9c6e10fbe7641590049a510555519190ab4689bb'
requirements_url = f'https://raw.githubusercontent.com/EF-Code/laliga-match-forecasting/{repo_revision}/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', requirements_url], check=True)

repo_dir = Path('/content/laliga-match-forecasting')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(['git', 'clone', '-q', '--no-checkout', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '-q', '--detach', repo_revision], check=True)
os.chdir(repo_dir)

In [ ]:
# Build the dataset locally. This step does not require Hugging Face authentication.
import importlib
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

importlib.invalidate_caches()
sys.path.insert(0, '/content/laliga-match-forecasting/src')
import laliga_forecasting.build_dataset as builder
builder = importlib.reload(builder)

build_log = StringIO()
try:
    with redirect_stdout(build_log), redirect_stderr(build_log):
        output_dir = Path('/content/laliga-output')
        matches = builder.fetch_all_seasons()
        pre_match, observations, team_stats = builder.build_pre_match_dataset(matches)
        statsbomb_bundle = builder.build_statsbomb_bundle(output_dir / 'statsbomb-cache')
        paths = builder.write_outputs(output_dir, pre_match, observations, team_stats, statsbomb_bundle)
    print('DATASET_BUILT')
    print(f'MATCH_ROWS {len(matches)}')
    print(f'PRE_MATCH_ROWS {len(pre_match)}')
    print(f'TEAM_MATCH_ROWS {len(team_stats)}')
    for artifact_name, frame in statsbomb_bundle.items():
        print(f'{artifact_name.upper()}_ROWS {len(frame)}')
finally:
    build_log.close()
    del build_log

## Optional: publish to Hugging Face

Set `PUBLISH_TO_HUB = True` only when you want to publish the generated dataset under your own Hugging Face account. The cell reads `HF_TOKEN` from Colab Secrets without prompting for or printing it. Leave the default as `False` to build and run the baseline without any Hugging Face credentials.

In [ ]:
PUBLISH_TO_HUB = False

if not PUBLISH_TO_HUB:
    print('HF_DATASET_PUBLISH_SKIPPED')
else:
    from contextlib import redirect_stderr, redirect_stdout
    from google.colab import userdata
    from io import StringIO

    hf_token = None
    publish_log = StringIO()
    try:
        hf_token = userdata.get('HF_TOKEN')
        if not hf_token:
            raise RuntimeError('Add a write-capable HF_TOKEN secret in Colab and enable notebook access')
        with redirect_stdout(publish_log), redirect_stderr(publish_log):
            builder.publish_to_hub(output_dir, pre_match, paths, token=hf_token)
        print('DATASET_PUBLISHED')
    finally:
        del hf_token
        publish_log.close()
        del publish_log

In [ ]:
!PYTHONPATH=src python -m laliga_forecasting.train_baseline --input /content/laliga-output/pre_match_forecasting.parquet --output /content/laliga-output/baseline_metrics.json